In [14]:
# Standard library imports
from pathlib import Path

# Third-party library imports
from google.colab import drive
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing import image

In [15]:
# Mount Google Drive and define paths to access the model weights
drive.mount('/content/drive')
BASE_DIR = Path("/content/drive/MyDrive/PokerCardClassification")
WEIGHTS_PATH = BASE_DIR / "best_card_model_weights.weights.h5"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Configure model constants
IMG_SIZE = (224, 224)
NUM_CLASSES = 53

In [17]:
# Reconstruct the model architecture
base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
model_eff = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [18]:
# Load the trained weights into the reconstructed model architecture
model_eff.load_weights(WEIGHTS_PATH)

In [19]:
# Map model prediction indices to class names
class_labels = [
    "ace_clubs", "ace_diamonds", "ace_hearts", "ace_spades",
    "eight_clubs", "eight_diamonds", "eight_hearts", "eight_spades",
    "five_clubs", "five_diamonds", "five_hearts", "five_spades",
    "four_clubs", "four_diamonds", "four_hearts", "four_spades",
    "jack_clubs", "jack_diamonds", "jack_hearts", "jack_spades",
    "joker",
    "king_clubs", "king_diamonds", "king_hearts", "king_spades",
    "nine_clubs", "nine_diamonds", "nine_hearts", "nine_spades",
    "queen_clubs", "queen_diamonds", "queen_hearts", "queen_spades",
    "seven_clubs", "seven_diamonds", "seven_hearts", "seven_spades",
    "six_clubs", "six_diamonds", "six_hearts", "six_spades",
    "ten_clubs", "ten_diamonds", "ten_hearts", "ten_spades",
    "three_clubs", "three_diamonds", "three_hearts", "three_spades",
    "two_clubs", "two_diamonds", "two_hearts", "two_spades"
]

In [20]:
def predict_single_image(img_path_str):
    """Loads a single image, verifies its extension, processes it, and predicts."""
    img_path = Path(img_path_str)
    VALID_EXTENSIONS = [".jpg", ".jpeg", ".png"]

    # File extension validation
    if img_path.suffix.lower() not in VALID_EXTENSIONS:
        print(f"\n [ERROR]: Invalid file format. Must be one of the following: {', '.join(VALID_EXTENSIONS)}\n")
        return

    # File existence validation
    if not img_path.exists():
        print(f"\n [ERROR]: The file does not exist at the specified path: {img_path_str}\n")
        return

    # Load the image from the path and resize it to match the model's expected input dimensions
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)

    # Transform the single image array into a batch format
    img_array = np.expand_dims(img_array, axis=0)

    # Scale and normalize the pixel values to match the specific distribution used during EfficientNet training
    processed_img = preprocess_input(img_array)

    # The model predicts class probabilities
    predictions = model_eff.predict(processed_img, verbose=0)

    # Find the index position of the highest probability
    class_idx = np.argmax(predictions, axis=1)[0]

    # Extract the highest probability score
    confidence = predictions[0][class_idx] * 100

    # Map the numerical index to its corresponding text string label
    predicted_card = class_labels[class_idx]

    # Display prediction results
    print("\n================ PREDICTION RESULT ================")
    print(f"Predicted Class: {predicted_card}")
    print(f"Confidence:       {confidence:.2f}%")
    print("===================================================\n")

In [21]:
# Prompt the user for input and execute the prediction
test_image_path = input("Please enter the path to the card image: ")
predict_single_image(test_image_path)

Please enter the path to the card image: /content/ace_clubs.png

================ PREDICTION RESULT ================
Predicted Class: ace_clubs
Confidence:       93.32%

